In [1]:
from networks_update import *

from torch.utils.data import Dataset
import torch
import SimpleITK as sitk
import os
import numpy as np
import matplotlib.pyplot as plt
from torch.utils.data import DataLoader
import torch
import torch.nn as nn
import torch.optim as optim
from networks_update import *
import csv
import time
import torchvision.utils as vutils
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap, BoundaryNorm


In [2]:
# HARDCORE FOR NOW
two_d = False # NOTE: 2D vs 3D
n_downsample = 3
dim=2
n_res = 1
img_x = 128
img_y = 120
img_z = 120
latent_dim = 2000


# Dataset path
train_folder = "./datasets/3d_anat_align/human_train/"
train_paths = [train_folder + file_name for file_name in os.listdir(train_folder)]

train_paths = train_paths

val_folder = "./datasets/3d_anat_align/human_test/"
val_paths = [val_folder + file_name for file_name in os.listdir(val_folder)]
val_paths.sort()

# folder to save model checkpoints
train_save_folder = "./VAE_train/3d_anat/human_train_testing/"

os.makedirs(train_save_folder + "/test_images", exist_ok=True)

# file to save losses
csv_file = 'loss_log.csv'

# epoch to load from
start_epoch = 0

# num epoch to train for
num_epochs = 1000

# how often to save model checkpoints and images
save_imgs = True
save_imgs_freq = 2
save_model_freq = 5

lr=1e-3
batch_size= 2

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


In [3]:
### Updated to be VAE style
class Encoder_VAE(nn.Module):
    def __init__(self, n_downsample, n_res, input_dim, dim, norm, activ, pad_type):
        super(Encoder_VAE, self).__init__()
        self.model = []
        if two_d:
            self.model += [Conv2dBlock(input_dim, dim, 7, 1, 3, norm=norm, activation=activ, pad_type=pad_type)]
        else:
            self.model += [Conv3dBlock(input_dim, dim, 7, 1, 3, norm=norm, activation=activ, pad_type=pad_type)]

        # downsampling blocks
        for i in range(n_downsample):
            if two_d:
                self.model += [Conv2dBlock(dim, 2 * dim, 4, 2, 1, norm=norm, activation=activ, pad_type=pad_type)]
            else:
                self.model += [Conv3dBlock(dim, 2 * dim, 4, 2, 1, norm=norm, activation=activ, pad_type=pad_type)]
            dim *= 2

        # residual blocks
        self.model += [ResBlocks(n_res, dim, norm=norm, activation=activ, pad_type=pad_type)]

        # NOTE: extra to map down to lower dim
        #self.model += [nn.Flatten(), nn.Linear(flattened_dim, latent_dim), nn.LayerNorm(latent_dim), nn.ReLU()]

        self.output_dim = dim

        # Convolutional mean & logvar heads (preserve spatial structure!)
        if two_d:
            self.fc_mu     = nn.Conv2d(dim, dim, 1)
            self.fc_logvar = nn.Conv2d(dim, dim, 1)
        else:
            self.fc_mu     = nn.Conv3d(dim, dim, 1)
            self.fc_logvar = nn.Conv3d(dim, dim, 1)


        self.model = nn.Sequential(*self.model)

    def forward(self, x):
        out = self.model(x)
        means = self.fc_mu(out) # self.inplace(out)
        log_vars = self.fc_logvar(out) #self.inplace(out) # why was it called log vars? Probs cuz of how it's used in KL divergence
        return means, log_vars
    
class Reshape(nn.Module):
    def __init__(self):
        super().__init__()
        #self.shape = (16, 16, 15, 15)  # e.g., (-1, 512) or (batch_size, channels, height, width)
        self.shape = (dim*2**n_downsample, int(img_x/2**n_downsample), int(img_y/2**n_downsample), int(img_z/2**n_downsample))  # e.g., (-1, 512) or (batch_size, channels, height, width)

    def forward(self, x):
        return x.reshape(x.size(0), *self.shape)  # keeps batch dim intact

class Decoder_VAE(nn.Module):
    def __init__(self, n_upsample, n_res, dim, output_dim, res_norm='in', activ='relu', pad_type='zero'): # NOTE: updated normalization to in to not have to compute weight and bias externally
        super(Decoder_VAE, self).__init__()

        self.model = []

        #self.model += [nn.Linear(latent_dim, flattened_dim), nn.LayerNorm(flattened_dim), nn.ReLU(), Reshape()]
 
        # AdaIN residual blocks # NOTE: changed!!
        self.model += [ResBlocks(n_res, dim, res_norm, activ, pad_type=pad_type)]

        # upsampling blocks
        for i in range(n_upsample):
            if two_d:
                self.model += [nn.Upsample(scale_factor=2),
                            Conv2dBlock(dim, dim // 2, 5, 1, 2, norm='in', activation=activ, pad_type=pad_type)] # NOTE: could update to instance norm since only a batch size of 2 -> don't want to normalize over full layer??
            else:
                self.model += [nn.Upsample(scale_factor=2),
                    Conv3dBlock(dim, dim // 2, 5, 1, 2, norm='in', activation=activ, pad_type=pad_type)] # NOTE: could update to instance norm since only a batch size of 2 -> don't want to normalize over full layer??
            
            dim //= 2
        # use reflection padding in the last conv layer
        if two_d:
            self.model += [Conv2dBlock(dim, output_dim, 7, 1, 3, norm='none', activation='none', pad_type=pad_type)] 
        else:
            self.model += [Conv3dBlock(dim, output_dim, 7, 1, 3, norm='none', activation='none', pad_type=pad_type)] 

        # use reflection padding in the last conv layer
        self.model = nn.Sequential(*self.model)

    def forward(self, x):
        return self.model(x)
    



# Loss function for VAE
def loss_func(imgs, recons, means, log_vars):
    criterion = nn.CrossEntropyLoss()
    recon = criterion(recons, imgs) # computes average per voxel (in CVAE they use this instead to sum over all voxels)

    BS = batch_size
    num_voxels = 120*120*128 # NOTE: update for 3D
    #num_voxels = 64*64*64 # NOTE: update for 2D
    beta = 10
    KLD = (-0.5 * torch.sum(1 + log_vars - means.pow(2) - log_vars.exp())) / (num_voxels * BS)

    return recon + beta*KLD

def reparameterization(means, log_vars):
    # move random vars sampled from a normal dist to size log_vars to device
    epsilon = torch.randn_like(log_vars).to(device) 
    std = torch.exp(0.5 * log_vars)
    z = means + std * epsilon
    return z

class Segmentation3DDataset(Dataset):
    def __init__(self, image_paths, transform=None):
        self.image_paths = image_paths
        self.transform = transform  # Optional (e.g., normalization, crop)

    def __len__(self):
        return len(self.image_paths)

    def __getitem__(self, idx):
        image = sitk.ReadImage(self.image_paths[idx])
        image = sitk.GetArrayFromImage(image)

        image = np.expand_dims(image, axis=0)
        if self.transform:
            image = self.transform(image)

        return torch.from_numpy(image).to(torch.float)

two_d = False

# ===== Your consistent color setup =====
base_colors = plt.cm.get_cmap('tab20').colors  # 20 RGBA colors
n_labels = 300
repeated_colors = np.tile(base_colors, (n_labels // 20 + 1, 1))[:n_labels]
cmap = ListedColormap(repeated_colors)
n_labels = 270
norm = BoundaryNorm(np.arange(n_labels + 1), cmap.N)

# Manually set label 0 to white
colors_with_white_bg = cmap.colors
colors_with_white_bg[0] = (1.0, 1.0, 1.0)  # RGB white
cmap = ListedColormap(colors_with_white_bg)

def __write_images(image_outputs, display_image_num, file_name):
    imgs, recons = image_outputs

    # If 3D volumes: [B, D, H, W] → take middle slice
    if imgs.ndim == 4:
        slice_idx = imgs.shape[1] // 2
        imgs = torch.stack([img[slice_idx] for img in imgs[:display_image_num]])
        recons = torch.stack([img[slice_idx] for img in recons[:display_image_num]])
    else:
        imgs = imgs[:display_image_num]
        recons = recons[:display_image_num]

    # Apply discrete colormap and return RGB tensors
    def apply_cmap_rgb(tensor):
        arr = tensor.cpu().numpy().astype(np.int32)
        rgb_list = []
        for img in arr:
            rgb_img = cmap(norm(img))[..., :3]  # drop alpha
            rgb_tensor = torch.from_numpy(rgb_img).permute(2, 0, 1)  # [3, H, W]
            rgb_list.append(rgb_tensor)
        return torch.stack(rgb_list)

    imgs_rgb = apply_cmap_rgb(imgs)
    recons_rgb = apply_cmap_rgb(recons)

    # Create grids
    grid_in = vutils.make_grid(imgs_rgb, nrow=display_image_num, padding=2)
    grid_rec = vutils.make_grid(recons_rgb, nrow=display_image_num, padding=2)

    # Stack vertically
    full_grid = torch.cat([grid_in, grid_rec], dim=1)

    vutils.save_image(full_grid, file_name)

class EarlyStopping:
    def __init__(self, patience=10, verbose=True, save_path="best_model.pth"):
        """
        patience: how many epochs to wait after last improvement
        verbose: print updates
        save_path: where to save the best model
        """
        self.patience = patience
        self.counter = 0
        self.best_loss = np.inf
        self.early_stop = False
        self.verbose = verbose
        self.save_path = save_path

    def __call__(self, val_loss, model_dict):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            self.save_checkpoint(model_dict)
        else:
            self.counter += 1
            if self.verbose:
                print(f"EarlyStopping counter: {self.counter}/{self.patience}")
            if self.counter >= self.patience:
                self.early_stop = True

    def save_checkpoint(self, model_dict):
        if self.verbose:
            print(f"Validation loss improved → {self.best_loss:.4f}. Saving model...")
        torch.save(model_dict, self.save_path)

/tmp/ipykernel_967805/215811026.py:130: MatplotlibDeprecationWarning: The get_cmap function was deprecated in Matplotlib 3.7 and will be removed two minor releases later. Use ``matplotlib.colormaps[name]`` or ``matplotlib.colormaps.get_cmap(obj)`` instead.
  base_colors = plt.cm.get_cmap('tab20').colors  # 20 RGBA colors


In [4]:
# Create dataset and dataloader
train_dataset = Segmentation3DDataset(image_paths=train_paths)
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)

val_dataset = Segmentation3DDataset(image_paths=val_paths)
val_loader = DataLoader(val_dataset, batch_size=1, shuffle=False)

# Initialize model
encoder = Encoder_VAE(n_downsample=n_downsample, n_res=n_res, input_dim=1, dim=dim, norm='in', activ='relu', pad_type='zero') # encodes to 32 dim??
decoder = Decoder_VAE(n_upsample=n_downsample, n_res=n_res, dim=encoder.output_dim, output_dim=271)

# Move to GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
encoder.to(device)
decoder.to(device)

# if not training from 0 load a pretrained model from saved checkpoint
if start_epoch != 0:
    encoder.load_state_dict(torch.load(train_save_folder + "checkpoint_epoch_" + str(start_epoch))['encoder_state_dict'])
    decoder.load_state_dict(torch.load(train_save_folder + "checkpoint_epoch_" + str(start_epoch))['decoder_state_dict'])

# Loss function
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=lr)

save_loss = []

# for saving images during training
display_size = 16 # num images to display

early_stopping = EarlyStopping(patience=5, save_path=train_save_folder + "best_model.pth")


print("Start training!!")
for epoch in range(start_epoch, start_epoch + num_epochs + 1):
    epoch_start_time = time.time()
    encoder.train()
    decoder.train()
    train_loss = 0

    for batch in train_loader:
        batch = batch.to(device)  # (B, C, D, H, W)

        # Forward pass
        means, log_vars = encoder(batch)

        # get latent vectors - sampled from learned dists
        z = reparameterization(means, log_vars)

        recon = decoder(z)

        # Compute loss
        loss = loss_func(batch.squeeze(1).long(), recon, means, log_vars)


        # Backprop
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        train_loss += loss.item()

    encoder.eval()
    decoder.eval()
    val_loss = 0

    img_to_save = []
    recon_to_save = []

    with torch.no_grad():
        for i, val_batch in enumerate(val_loader):
            val_batch = val_batch.to(device)

            means, log_vars = encoder(val_batch)
            # get latent vectors - sampled from learned dists
            z = reparameterization(means, log_vars)
            recon = decoder(z)
            # Compute loss
            loss = loss_func(val_batch.squeeze(1).long(), recon, means, log_vars)

            val_loss += loss.item()

            # save a few test images
            if save_imgs and (epoch % save_imgs_freq == 0) and i < 3:
                recon = torch.argmax(recon, dim=1).squeeze().detach().cpu()
                recon_to_save.append(recon) # might just be a shallow copy

                #recon_img = sitk.GetImageFromArray(np.array(recon))
                # save a few nifty image reconstructions - use for analysis
                #sitk.WriteImage(recon_img, train_save_folder + "test_images/recon_" + val_paths[i].split("/")[-1].split(".")[0] + "_epoch_" + str(epoch) + ".nii")

                # to display
                img_to_save.append(val_batch)

            if save_imgs and i >= 3 and i < display_size and (epoch % save_imgs_freq == 0):
                recon = torch.argmax(recon, dim=1).squeeze().detach().cpu()
                recon_to_save.append(recon)
                img_to_save.append(val_batch)

        # save a png of some reconstructions - to observe during training
        if save_imgs and epoch % save_imgs_freq == 0:
            img_to_save = torch.stack(img_to_save).squeeze()
            recon_to_save = torch.stack(recon_to_save)

            print("save images...")
            __write_images([img_to_save, recon_to_save], display_size, train_save_folder + "test_images/recons_epoch_" + str(epoch) + ".png")


    train_loss = train_loss / len(train_loader)
    val_loss = val_loss / len(val_loader)
    elapsed_time = time.time() - epoch_start_time
    print(f"Epoch [{epoch+1}/{num_epochs}], Train loss: {train_loss:.4f}, Val loss: {val_loss:.4f} (time: {elapsed_time:.4f})")
    
    # write loss to a csv every epoch
    with open(train_save_folder + 'loss_log.csv', mode='a', newline='') as file:
        writer = csv.writer(file)
        writer.writerow([epoch, train_loss, val_loss])  # Writes a single row with two values
   
    if epoch % save_model_freq == 0:
        # model dict for saving
        checkpoint = {
            'epoch': epoch,
            'encoder_state_dict': encoder.state_dict(),
            'decoder_state_dict': decoder.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'train_loss': train_loss,
            'test_loss': val_loss,
        }

        # check early stopping
        early_stopping(val_loss, checkpoint)
        if early_stopping.early_stop:
            print("Early stopping triggered. Stopping training.")
            break

Start training!!
save images...
Epoch [1/1000], Train loss: 0.9625, Val loss: 0.5804 (time: 47.4318)
Validation loss improved → 0.5804. Saving model...
Epoch [2/1000], Train loss: 0.5654, Val loss: 0.5449 (time: 46.9199)
save images...
Epoch [3/1000], Train loss: 0.5301, Val loss: 0.5083 (time: 46.9768)
Epoch [4/1000], Train loss: 0.4957, Val loss: 0.4719 (time: 46.9761)
save images...
Epoch [5/1000], Train loss: 0.4623, Val loss: 0.4408 (time: 47.0677)
Epoch [6/1000], Train loss: 0.4339, Val loss: 0.4136 (time: 46.9221)
Validation loss improved → 0.4136. Saving model...
save images...
Epoch [7/1000], Train loss: 0.4108, Val loss: 0.3910 (time: 47.1157)
Epoch [8/1000], Train loss: 0.3906, Val loss: 0.3730 (time: 46.9585)
save images...
Epoch [9/1000], Train loss: 0.3728, Val loss: 0.3564 (time: 47.1387)
Epoch [10/1000], Train loss: 0.3583, Val loss: 0.3443 (time: 47.0356)
save images...
Epoch [11/1000], Train loss: 0.3450, Val loss: 0.3308 (time: 47.1328)
Validation loss improved → 0.3

KeyboardInterrupt: 